# Manual Marker-Based Annotation

Visualizes curated marker gene panels on UMAP for manual cell-type
annotation, ranks cluster marker genes (logistic regression and TF-IDF),
and highlights specific sample/response subgroups on the UMAP.

## Setup

In [ ]:
import os

os.environ["NUMBA_CACHE_DIR"] = "/path/to/tmp"
os.environ["MPLCONFIGDIR"] = "/path/to/tmp"

import anndata
import scanpy as sc
import episcanpy as epi
from sklearn.feature_extraction.text import TfidfTransformer

sc.settings.verbosity = 1  # errors (0), warnings (1), info (2), hints (3)
sc.settings.figdir = "./figures-cellbender/integration/3_annotation/1_broad/"
sc.settings.set_figure_params(dpi=80)

## Config

In [ ]:
input_path = "/path/to/output/S4_scVI_integrated_adata_cluster_latent10.h5ad"

groupby_col = "leiden_scvi"

# curated marker gene panels for manual annotation. Each is plotted on its
# own UMAP figure, colored by the given genes plus a few metadata columns.
marker_panels = {
    "scvi_sanitycheck": dict(
        genes=["NAPSA", "NKX2-1", "CEACAM5", "CEACAM1", "NTRK2", "TP63", "MALAT1", "NEAT1", "FTX", "ADK"],
        extra_color=["ID"],
    ),
    "liver_markers": dict(
        genes=[
            "AFP", "TTR", "CYP3A4", "ASGR1", "ASGR2",  # hepatocytes
            "KRT19", "SOX9", "CFTR",                    # cholangiocytes
            "LYVE1", "STAB1", "STAB2",                  # LSECs
            "CLEC4F", "MARCO", "CD68",                  # Kupffer cells
        ],
        extra_color=["ID"],
    ),
    "check_major_markers": dict(
        genes=[
            "KRT7", "KRT18", "EGFR",              # Epithelial
            "TP63", "KRT17", "LAMC2",             # Basal-like
            "EPCAM", "CD24",                      # ciliated Epithelial
            "AGER", "CLDN18",                     # Alveolar Type I
            "SCGB3A2", "SCGB3A1",                 # Club cell
            "BPIFB1", "SCGB3A1", "FOXA3", "SPDEF",# Goblet
            "SFTPC", "SFTPD",                     # Alveolar Type II
            "PTPRC",                              # Immune
            "CD69", "CD3E", "CD3G", "CD8A", "CD4",# T cell
            "FOXP3", "IL2RA",                     # Treg
            "FCGR3A", "CD14", "CD163",            # Macrophage
            "NCAM1", "NKG7",                      # NK cell
            "ACTA2", "COL6A2", "PDGFRB", "MMP2",  # Fibroblast/stroma
            "CD34", "PECAM1",                     # Endothelial
            "EBF1", "MS4A1",                      # B cell
        ],
        extra_color=[groupby_col],
        ncols=8,
    ),
    "cellcycle_emt_wnt_markers": dict(
        genes=[
            "E2F2", "CENPF", "NCAPG", "SPC25",             # cell cycle
            "RFC3", "CLSPN", "EXO1", "FANCB",              # DNA repair
            "CDH2", "NCAM1", "FOXM1", "FOXG1", "ASTN1", "TWIST1", "TWIST2", "ZEB1", "SNAI2",  # EMT
            "SYN1", "INSM1", "TAGLN3", "SEZ6", "KCNC1",    # neuronal differentiation
            "WNK2", "HIST1H4C", "ASPM", "FZD3", "TCF4",    # Wnt signaling
        ],
        extra_color=["sample", groupby_col, "group2"],
    ),
    "gene_1": dict(
        genes=[
            "ALDH2", "SIRT1", "ALDH1A1", "SOX2", "MYC", "EZH2", "NAPSA", "SFTA2", "NKX2-1",
            "CD44", "ABCG2", "LGALS1", "CHGA", "CHGB", "NEUROD1", "ASCL1", "POU2F3", "YAP1",
            "GRP", "KRT5", "TP63", "ALPP", "CEACAM5", "CEACAM1",
        ],
        extra_color=["ID", groupby_col, "group2"],
    ),
    "gene_2_cytokines_exhaustive": dict(
        genes=[
            "IL1B", "CCL5", "IL4", "IL13", "CCL22", "IFNG", "CXCL12", "CXCL13", "CXCL9", "CXCL10", "IL2",  # cytokines/chemokines
            "GZMA", "GZMB", "GZMK", "GNLY", "PRF1",         # effectors
            "TCF7", "LEF1", "CCR7",                          # naive
            "LAG3", "TIGIT", "PDCD1", "HAVCR2", "CTLA4", "IDO1", "PDCD1LG2", "CD274", "CD276",  # inhibitory receptors
            "CD28", "ICOS", "TNFRSF14", "TNFRSF9",           # costimulatory
            "HIF1A", "ID2", "EOMES", "HOPX", "TBX21",        # transcription factors
            "CD96", "PVR",                                    # FAO
        ],
        extra_color=["ID", "group2", groupby_col],
    ),
    "gene_3_cytokines_exhaustive": dict(
        genes=["IL13RA1", "IL4R", "CD47", "CD27", "CMKLR1", "CXCR6", "PSMB10", "STAT1", "CD38", "ENTPD1", "MUC1", "CD28"],
        extra_color=["ID", "group2", groupby_col],
    ),
}


## Load data

In [ ]:
adata = sc.read(input_path)
adata

## Cluster overview

In [ ]:
sc.pl.umap(adata, color=[groupby_col], legend_fontsize="x-small", size=6)
sc.pl.umap(
    adata, color=[groupby_col], size=15, legend_loc="on data",
    legend_fontsize="x-small", legend_fontoutline=True,
)

## Visualize marker gene panels

In [ ]:
for panel_name, panel in marker_panels.items():
    sc.pl.umap(
        adata, color=panel["genes"] + panel["extra_color"],
        use_raw=False, color_map="OrRd", size=10,
        ncols=panel.get("ncols", 4), save=f"_{panel_name}.pdf",
    )

## Rank marker genes (logistic regression)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby_col, method="logreg", key_added="logreg")
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False, key="logreg", save="_logreg.pdf")
sc.pl.rank_genes_groups_dotplot(
    adata, n_genes=5, key="logreg", groupby=groupby_col,
    standard_scale="var", dendrogram=False, save="_logReg.pdf",
)

## Rank marker genes (TF-IDF)

In [ ]:
def run_tfidf(adata, groupby, method="wilcoxon"):
    """Binarize expression, compute TF-IDF, and rank genes per group."""
    counts = anndata.AnnData(X=adata.X, var=adata.var, obs=adata.obs, asview=False)
    adata_bin = epi.pp.binarize(counts, copy=True)
    transformer = TfidfTransformer()
    adata_bin.X = transformer.fit_transform(adata_bin.X.T).T
    sc.tl.rank_genes_groups(adata_bin, groupby=groupby, use_raw=False, method=method)
    return adata_bin

In [ ]:
adata_tfidf = run_tfidf(adata, groupby=groupby_col, method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata_tfidf, n_genes=10, dendrogram=False, standard_scale="var")
sc.pl.rank_genes_groups_dotplot(adata_tfidf, n_genes=5, dendrogram=False, standard_scale="var", save="_TF_IDF.pdf")

## Highlight specific groups on UMAP

For each `obs` column in `highlight_groups`, highlight each listed value individually.

In [ ]:
for obs_col, group_values in highlight_groups.items():
    for group_value in group_values:
        sc.pl.umap(
            adata, color=[obs_col], groups=[group_value], size=15,
            legend_fontsize="x-small", legend_fontoutline=True,
        )